# Aspect-level report generation — FLAN-T5 baseline + alternative LLM

Consumes `aspect_stats.json` (produced by `notebooks/aspect_stats_semeval_laptop.ipynb`,
Section 6 there) and turns the per-aspect sentiment table into a short natural-language
report, the deliverable described in `plans/task.txt` (Tuần 4 — Hoàng/Vinh/Hưng).

Two generators are compared on the **same facts**:

1. **FLAN-T5** (`google/flan-t5-large`) — the model originally specified for this step.
   In practice (per initial experiments) it tends to produce short, generic,
   template-like sentences and sometimes drops or garbles the numbers, likely because
   it is an instruction-tuned *encoder-decoder* model trained mostly on short
   classification/QA-style tasks rather than longer structured report writing.
2. **A proposed alternative: `microsoft/Phi-3-mini-4k-instruct`** (3.8B, decoder-only,
   chat-instruction-tuned). Rationale for this choice over FLAN-T5:
   - Much stronger instruction-following on multi-sentence, structured writing tasks
     (this is a documented gap for FLAN-T5 relative to modern chat-tuned decoder LLMs
     of similar or even smaller size).
   - Small enough (3.8B) to run in fp16 on a single Kaggle T4/P100 GPU without needing
     aggressive quantization, so inference stays simple and fast for a whole aspect
     table.
   - MIT-licensed, so it's safe to use and cite in a course project.
   - If a larger model is preferred and quantization is acceptable, `Qwen2.5-7B-Instruct`
     or `Mistral-7B-Instruct-v0.3` (loaded in 4-bit via `bitsandbytes`) are drop-in
     alternatives — just change `ALT_MODEL_NAME` in Section 4 below.

Both generators are prompted with **the exact same fact table** (positive/negative/
neutral counts + majority sentiment per aspect) and instructed to only use those
numbers. A lightweight **factual checker** (Section 6) then re-derives, for each aspect
mentioned in the generated text, whether the report's claimed sentiment matches the
table and flags any numbers that don't correspond to a real count in the table —
the same idea `aspect_stats_semeval_laptop.ipynb` already does for gold-vs-predicted
agreement, but applied to *generated text vs. source facts* here.

Kaggle setup: add the Kaggle Dataset/Model containing `aspect_stats.json` (the output
of `aspect_stats_semeval_laptop.ipynb`, uploaded as its own dataset — or simply re-run
that notebook as a preceding cell/pipeline input) as this notebook's input, enable a
GPU accelerator (needed for Section 4 — the Phi-3 generation; FLAN-T5-large alone
could run on CPU but is slow).

In [4]:
!pip install -q -U transformers accelerate sentencepiece bitsandbytes

In [5]:
import glob
import json
import re
from collections import Counter
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

SENTIMENT_LABELS = ("positive", "negative", "neutral")
TOP_N_ASPECTS = 12  # keep the report focused; avoids overly long/rambling prompts
MIN_MENTIONS_FOR_REPORT = 2  # aspect_stats.json was already filtered with this cutoff

## 1. Locate `aspect_stats.json`

Same pattern as the model/data locators in `aspect_stats_semeval_laptop.ipynb`:
search likely Kaggle input paths and local fallbacks, and fail with a clear message
telling the user what to attach if nothing is found.

In [6]:
def find_aspect_stats_file():
    search_roots = [
        "/kaggle/input",
        "/kaggle/working",
        "../results",
        "results",
        "../output",
        "output",
    ]
    candidates = []
    for root in search_roots:
        candidates.extend(glob.glob(f"{root}/**/aspect_stats.json", recursive=True))
        candidates.extend(glob.glob(f"{root}/aspect_stats.json"))
    candidates = sorted(set(Path(c) for c in candidates), key=str)
    if not candidates:
        raise FileNotFoundError(
            "aspect_stats.json not found. Run notebooks/aspect_stats_semeval_laptop.ipynb "
            "first and either add its /kaggle/working output as this notebook's input "
            "(new Kaggle Dataset from notebook output), or place a local copy under "
            "results/aspect_stats.json."
        )
    if len(candidates) > 1:
        print(f"Found {len(candidates)} candidate file(s), using the first: {candidates}")
    return candidates[0]


STATS_PATH = find_aspect_stats_file()
print("Using aspect stats file:", STATS_PATH)

with open(STATS_PATH) as f:
    stats = json.load(f)

print("num_examples:", stats["num_examples"])
print("per_example_agreement:", stats["per_example_agreement"])
print("majority_sentiment_agreement:", stats["majority_sentiment_agreement"])
print("aspects (predicted table):", len(stats["predicted"]))

Using aspect stats file: output/aspect_stats.json
num_examples: 2313
per_example_agreement: 0.8617
majority_sentiment_agreement: 0.9012
aspects (predicted table): 15


## 2. Select the facts to report on

Use the **predicted** table — per the source notebook, this is the branch that
generalizes to unlabeled data (e.g. the Amazon Reviews demo in Tuần 5). Sort by
mention count (already the sort order saved in the JSON, but re-sort defensively)
and keep the top `TOP_N_ASPECTS` — a report covering every one of ~250 aspects
would not be a "short summary report", and models tend to get worse and more
repetitive on very long structured inputs.

In [7]:
predicted_table = [
    row for row in stats["predicted"] if row["total"] >= MIN_MENTIONS_FOR_REPORT
]
predicted_table.sort(key=lambda r: (-r["total"], r["aspect"]))
report_facts = predicted_table[:TOP_N_ASPECTS]

for r in report_facts:
    print(
        f"  {r['aspect']:<20} total={r['total']:>3}  "
        f"+{r['positive']:<3} -{r['negative']:<3} ~{r['neutral']:<3}  "
        f"majority={r['majority_sentiment']}"
    )

  screen               total= 60  +32  -24  ~4    majority=positive
  price                total= 56  +49  -4   ~3    majority=positive
  use                  total= 53  +48  -4   ~1    majority=positive
  battery life         total= 52  +29  -17  ~6    majority=positive
  keyboard             total= 50  +25  -20  ~5    majority=positive
  battery              total= 47  +9   -34  ~4    majority=negative
  programs             total= 37  +17  -10  ~10   majority=positive
  features             total= 35  +26  -4   ~5    majority=positive
  software             total= 33  +8   -16  ~9    majority=negative
  warranty             total= 31  +2   -10  ~19   majority=neutral
  hard drive           total= 30  +3   -18  ~9    majority=negative
  windows              total= 30  +3   -16  ~11   majority=negative


## 3. Build the shared fact block + prompts

Both models see the **identical** bullet-point fact list, so any difference in the
generated report is attributable to the model, not the input. The instruction is
explicit about not inventing aspects, numbers, or sentiments beyond what's given —
this is what the factual checker in Section 6 verifies.

In [8]:
def format_facts_block(facts):
    lines = []
    for r in facts:
        lines.append(
            f"- {r['aspect']}: {r['total']} mentions total "
            f"({r['positive']} positive, {r['negative']} negative, {r['neutral']} neutral), "
            f"overall sentiment: {r['majority_sentiment']}"
        )
    return "\n".join(lines)


FACTS_BLOCK = format_facts_block(report_facts)
print(FACTS_BLOCK)

INSTRUCTION = (
    "You are writing a short customer-feedback summary report for a laptop product team. "
    "You are given per-aspect sentiment statistics derived from customer reviews. "
    "Write a concise report (5-8 sentences) that: "
    "(1) highlights the aspects customers feel most positively about, "
    "(2) highlights the aspects with the most negative feedback, "
    "(3) mentions any aspects with mixed/neutral feedback, "
    "and (4) where useful, cites the mention counts. "
    "Use ONLY the aspects, counts, and sentiments given below — do not invent, rename, "
    "or estimate any aspect, number, or sentiment that is not explicitly listed.\n\n"
    f"Aspect statistics:\n{FACTS_BLOCK}\n\nReport:"
)

print("\n--- Full instruction/prompt ---\n")
print(INSTRUCTION)

- screen: 60 mentions total (32 positive, 24 negative, 4 neutral), overall sentiment: positive
- price: 56 mentions total (49 positive, 4 negative, 3 neutral), overall sentiment: positive
- use: 53 mentions total (48 positive, 4 negative, 1 neutral), overall sentiment: positive
- battery life: 52 mentions total (29 positive, 17 negative, 6 neutral), overall sentiment: positive
- keyboard: 50 mentions total (25 positive, 20 negative, 5 neutral), overall sentiment: positive
- battery: 47 mentions total (9 positive, 34 negative, 4 neutral), overall sentiment: negative
- programs: 37 mentions total (17 positive, 10 negative, 10 neutral), overall sentiment: positive
- features: 35 mentions total (26 positive, 4 negative, 5 neutral), overall sentiment: positive
- software: 33 mentions total (8 positive, 16 negative, 9 neutral), overall sentiment: negative
- warranty: 31 mentions total (2 positive, 10 negative, 19 neutral), overall sentiment: neutral
- hard drive: 30 mentions total (3 positiv

## 4. Baseline: FLAN-T5

`google/flan-t5-large` (encoder-decoder, instruction-tuned). Included as the
originally-specified generator and as the point of comparison for the alternative
model in Section 5.

In [9]:
FLAN_MODEL_NAME = "google/flan-t5-large"

device = "cuda" if torch.cuda.is_available() else "cpu"

flan_tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME).to(device)
flan_model.eval()

flan_inputs = flan_tokenizer(INSTRUCTION, return_tensors="pt", truncation=True, max_length=1024).to(device)

with torch.no_grad():
    flan_output_ids = flan_model.generate(
        **flan_inputs,
        max_new_tokens=220,
        min_new_tokens=60,
        num_beams=4,
        no_repeat_ngram_size=3,
        length_penalty=1.1,
    )

flan_report = flan_tokenizer.decode(flan_output_ids[0], skip_special_tokens=True)
print("=== FLAN-T5 report ===\n")
print(flan_report)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== FLAN-T5 report ===

The screen is the most positive aspect of the laptop. The battery life is the least positive aspect. The keyboard has the most negative reviews. The warranty is the worst. The hard drive has the worst reviews. Overall, the laptop is a good product. It has a lot of positive reviews.


Free the FLAN-T5 model from GPU memory before loading the alternative model
(useful on single-GPU Kaggle sessions with limited VRAM).

In [10]:
del flan_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Proposed alternative: Phi-3-mini-4k-instruct

Decoder-only, chat-instruction-tuned, 3.8B parameters. Loaded in fp16 directly (fits
comfortably on a single T4/P100); switch `ALT_MODEL_NAME` below to
`"Qwen/Qwen2.5-7B-Instruct"` or `"mistralai/Mistral-7B-Instruct-v0.3"` (with
`load_in_4bit=True`) for a larger alternative if more VRAM is available.

In [ ]:
# ALT_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# alt_tokenizer = AutoTokenizer.from_pretrained(ALT_MODEL_NAME, trust_remote_code=True)
# alt_model = AutoModelForCausalLM.from_pretrained(
#     ALT_MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     trust_remote_code=True,
# )
# alt_model.eval()

# chat = [
#     {
#         "role": "system",
#         "content": "You are a precise data-reporting assistant. You never invent facts.",
#     },
#     {"role": "user", "content": INSTRUCTION},
# ]
# alt_prompt = alt_tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
# alt_inputs = alt_tokenizer(alt_prompt, return_tensors="pt").to(alt_model.device)

# with torch.no_grad():
#     alt_output_ids = alt_model.generate(
#         **alt_inputs,
#         max_new_tokens=280,
#         do_sample=False,
#         num_beams=1,
#         temperature=None,
#         top_p=None,
#         repetition_penalty=1.1,
#         pad_token_id=alt_tokenizer.eos_token_id,
#     )

# alt_report_full = alt_tokenizer.decode(
#     alt_output_ids[0][alt_inputs["input_ids"].shape[1]:], skip_special_tokens=True
# )
# alt_report = alt_report_full.strip()
# print("=== Phi-3-mini-4k-instruct report ===\n")
# print(alt_report)

## 6. Factual checker

Re-derives, for each aspect in `report_facts`, whether the generated report's claim
about that aspect is consistent with the source table:

- **mentioned / not mentioned** — does the aspect name appear in the report at all?
- **sentiment match** — if a sentiment-bearing sentence about the aspect is found,
  does its polarity (via a small keyword lexicon) agree with `majority_sentiment`?
- **unverified numbers** — any integer written near an aspect mention that does not
  equal that aspect's `total`/`positive`/`negative`/`neutral` count is flagged as a
  possible fabricated statistic.

This is a heuristic keyword-based checker (not a trained NLI model) — good enough to
catch gross fabrication/contradiction for a course-project sanity check, matching the
"lightweight factual checker" scope described in the project proposal.

In [16]:
POSITIVE_WORDS = {
    "positive", "favorably", "favorable", "praised", "liked", "loved", "great",
    "good", "strong", "well-received", "satisfied", "impressed", "excellent",
}
NEGATIVE_WORDS = {
    "negative", "criticized", "complaints", "complaint", "poor", "weak",
    "disliked", "frustration", "frustrated", "issues", "problems", "unhappy",
    "disappointing", "disappointed",
}
NEUTRAL_WORDS = {"neutral", "mixed", "divided", "split", "varied"}


def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def infer_sentiment_from_sentence(sentence):
    tokens = set(re.findall(r"[a-zA-Z]+", sentence.lower()))
    hits = {
        "positive": bool(tokens & POSITIVE_WORDS),
        "negative": bool(tokens & NEGATIVE_WORDS),
        "neutral": bool(tokens & NEUTRAL_WORDS),
    }
    hit_labels = [label for label, hit in hits.items() if hit]
    if len(hit_labels) == 1:
        return hit_labels[0]
    return "unclear"  # none, or conflicting, keyword hits


def check_report(report_text, facts):
    sentences = split_sentences(report_text)
    valid_numbers = set()
    for r in facts:
        valid_numbers.update({r["total"], r["positive"], r["negative"], r["neutral"]})

    per_aspect_results = []
    for r in facts:
        aspect = r["aspect"]
        matched_sentences = [s for s in sentences if aspect in s.lower()]
        if not matched_sentences:
            per_aspect_results.append(
                {"aspect": aspect, "mentioned": False, "sentiment_check": "n/a", "flagged_numbers": []}
            )
            continue

        inferred = [infer_sentiment_from_sentence(s) for s in matched_sentences]
        inferred = [i for i in inferred if i != "unclear"]
        if not inferred:
            sentiment_check = "unclear"
        elif r["majority_sentiment"] in inferred:
            sentiment_check = "match"
        else:
            sentiment_check = "mismatch"

        flagged_numbers = []
        for s in matched_sentences:
            for num_str in re.findall(r"\b\d+\b", s):
                num = int(num_str)
                if num not in valid_numbers:
                    flagged_numbers.append(num)

        per_aspect_results.append(
            {
                "aspect": aspect,
                "mentioned": True,
                "sentiment_check": sentiment_check,
                "flagged_numbers": sorted(set(flagged_numbers)),
            }
        )

    summary = Counter(r["sentiment_check"] if r["mentioned"] else "not_mentioned" for r in per_aspect_results)
    total_flagged_numbers = sum(len(r["flagged_numbers"]) for r in per_aspect_results)
    return {
        "per_aspect": per_aspect_results,
        "summary_counts": dict(summary),
        "total_flagged_numbers": total_flagged_numbers,
        "num_aspects_checked": len(facts),
    }


flan_check = check_report(flan_report, report_facts)
# alt_check = check_report(alt_report, report_facts)

print("=== FLAN-T5 factual check ===")
print(json.dumps(flan_check["summary_counts"], indent=2))
print("Flagged (possibly fabricated) numbers:", flan_check["total_flagged_numbers"])

# print("\n=== Phi-3-mini-4k-instruct factual check ===")
# print(json.dumps(alt_check["summary_counts"], indent=2))
# print("Flagged (possibly fabricated) numbers:", alt_check["total_flagged_numbers"])

=== FLAN-T5 factual check ===
{
  "match": 2,
  "not_mentioned": 6,
  "mismatch": 2,
  "unclear": 2
}
Flagged (possibly fabricated) numbers: 0


## 7. Save results

Save both reports and their factual-check breakdowns to `/kaggle/working/`, so the
generated text, the facts it was conditioned on, and the checker's verdict are all
traceable together for the write-up.

In [17]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "generated_reports.json"
out_path.write_text(
    json.dumps(
        {
            "source_stats_file": str(STATS_PATH),
            "top_n_aspects": TOP_N_ASPECTS,
            "facts_used": report_facts,
            "models": {
                "flan_t5": {
                    "model_name": FLAN_MODEL_NAME,
                    "report": flan_report,
                    "factual_check": flan_check,
                },
                # "alternative": {
                #     "model_name": ALT_MODEL_NAME,
                #     "report": alt_report,
                #     "factual_check": alt_check,
                # },
            },
        },
        indent=2,
    )
)
print(f"Saved generated reports + factual checks to {out_path}")

Saved generated reports + factual checks to results/generated_reports.json


## Next steps

- If `alternative`'s factual-check summary shows meaningfully fewer mismatches/
  flagged numbers than `flan_t5` (expected, given the initial experiments motivating
  this notebook), that's the evidence to cite in the report/write-up for switching
  the report-generation step away from FLAN-T5.
- If the alternative model still fabricates numbers occasionally, consider:
  tightening the prompt (e.g. few-shot example of a correctly-grounded sentence),
  lowering `TOP_N_ASPECTS` further, or adding a repair step that re-prompts the model
  with the specific factual-checker complaints and asks it to fix them.
- For the Amazon Reviews demo (Tuần 5): the same `format_facts_block` /
  `INSTRUCTION` / `check_report` functions apply unchanged once the aspect-extraction
  step (still not built — see `plans/project-plan.md`, Tuần 4) produces a predicted
  aspect_stats.json for that dataset.